# Gemma 4 12B (QAT) — download + inference

Downloads a quantization-aware-trained Gemma 4 12B checkpoint and runs inference.
Runs on a Colab GPU runtime, locally, or headlessly via the Colab CLI.

Everything below was verified end-to-end on an A100-SXM4-40GB Colab runtime
(`transformers` 5.13.1, `torch` 2.11.0+cu128, `compressed-tensors` 0.18.0).

## Pick a variant by **runtime** memory, not download size

| Repo | Download | **VRAM to run** | Loader |
|---|---|---|---|
| `google/gemma-4-12B-it-qat-w4a16-ct` | 10.3 GB | **~26 GB** | `transformers` |
| `google/gemma-4-12B-it-qat-q4_0-unquantized` | ~24 GB | ~28 GB | `transformers` |
| `google/gemma-4-12B-it-qat-q4_0-gguf` | ~7 GB | ~8 GB | `llama.cpp` |

**The trap that actually bites.** `compressed-tensors` 0.18.0 does *not* run `pack-quantized`
weights in packed form. It registers a decompress hook that fires on the first forward pass and
expands every `Linear` back to BF16 on the GPU. So `w4a16-ct` **loads** at 8.3 GB and then
**runs** at ~25 GB peak. Passing `run_compressed=True` does not change this — that path still
calls `decompress_model`, so it OOMs identically.

Consequence: a 24 GB L4 loads this checkpoint fine and then dies on the first token. Measured
directly — `OutOfMemoryError: Tried to allocate 226.00 MiB ... 83.12 MiB is free`. The int4
serialization buys you download size and QAT-trained weight values, not a smaller resident model.

So the real cutoff is **~30 GB VRAM** for either `transformers` path, and anything smaller
(L4, T4, free tier, CPU) goes to the GGUF path in section 8.

A second naming trap: `-unquantized` is **BF16**. Those are weights *trained* to quantize well,
not weights that are quantized.

## Running this on Colab from a terminal

```bash
colab --auth=adc new -s gemma --gpu A100          # T4|L4|G4|A100|H100
colab --auth=adc install -s gemma compressed-tensors accelerate hf_transfer
colab --auth=adc exec -s gemma --timeout 2400 -f notebooks/gemma4_12b_qat_inference.ipynb
colab --auth=adc log  -s gemma -o run.md
colab --auth=adc stop -s gemma                    # nothing else reclaims the VM
```

`--auth=adc` goes **before** the subcommand. `--timeout 2400` is not optional: `colab exec`
defaults to **30 seconds** and the download alone exceeds that.

Kernel state persists between `colab exec` calls on one session, so you can also drive it
incrementally — load once, then send follow-up prompts as separate short calls:

```bash
echo 'print(generate([{"role":"user","content":"hi"}]))' | colab --auth=adc exec -s gemma --timeout 300
```

## 1. Inspect the runtime

In [ ]:
import shutil, subprocess, sys

if shutil.which("nvidia-smi"):
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip())
else:
    print("no nvidia-smi — CPU-only runtime")

print(f"python {sys.version.split()[0]}")
print(f"free disk: {shutil.disk_usage('/').free / 1e9:.0f} GB")

## 2. Install dependencies

Colab's image already ships a `transformers` that knows Gemma 4 (5.13.1 has `gemma4_unified` in
`CONFIG_MAPPING`). `compressed-tensors` is what reads the `w4a16-ct` weights and is *not*
preinstalled.

Under the Colab CLI, prefer `colab install -s <name> ...` from the terminal — it uses `uv` and is
much faster than `%pip` inside the kernel.

In [ ]:
%pip install -q -U "transformers>=5.13" torch accelerate compressed-tensors huggingface_hub hf_transfer

import transformers, torch
print("transformers", transformers.__version__)
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)

# Sanity-check that this build actually knows the architecture.
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print("gemma4_unified supported:", "gemma4_unified" in CONFIG_MAPPING_NAMES)

## 3. Pick a variant

Selection is driven by VRAM against the **runtime** footprint from the table above, so the
threshold is ~30 GB rather than the 10 GB the download size would suggest.

In [ ]:
import os, torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"          # faster multipart downloads
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation

W4A16 = "google/gemma-4-12B-it-qat-w4a16-ct"
BF16 = "google/gemma-4-12B-it-qat-q4_0-unquantized"
GGUF = "google/gemma-4-12B-it-qat-q4_0-gguf"

# Measured peak on A100-40GB with a short context; leave headroom for KV cache.
W4A16_RUNTIME_GB = 26.0

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    sm = props.major * 10 + props.minor
    print(f"{props.name}: {vram_gb:.0f} GB, sm{sm}")
else:
    vram_gb, sm = 0.0, 0
    print("no GPU visible")

if vram_gb >= 34:
    MODEL_ID = W4A16   # smaller download than BF16, same resident size after decompression
elif vram_gb >= 30:
    MODEL_ID = W4A16
    print("!! tight — expect OOM if you raise max_new_tokens or context length")
else:
    MODEL_ID = GGUF    # L4/T4/CPU: the transformers path decompresses to BF16 and will not fit

# Force a variant by hand here if you want, e.g. MODEL_ID = GGUF
USE_TRANSFORMERS = MODEL_ID != GGUF

print("selected:", MODEL_ID)
if not USE_TRANSFORMERS:
    print("\n>> sections 4-7 (transformers) skip themselves; section 8 does the work.")

## 4. Download the weights

The QAT checkpoints are Apache 2.0 and not gated, so no token is needed — but an anonymous
download is rate-limited, so set `HF_TOKEN` if you have one.

`*.jinja` in `allow_patterns` is load-bearing: `transformers` v5 stores the chat template in
`chat_template.jinja`, and omitting the pattern silently skips it, which breaks
`apply_chat_template` later.

In [ ]:
import os, time
from huggingface_hub import snapshot_download

# Colab's secret vault is only reachable from the web UI; under the CLI this times out.
# Guard against both the timeout and a None return (setdefault(None) would raise).
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ.setdefault("HF_TOKEN", _tok)
except Exception as e:
    print(f"(no Colab secret: {type(e).__name__}) — continuing anonymously")

if USE_TRANSFORMERS:
    t0 = time.time()
    local_dir = snapshot_download(
        MODEL_ID,
        allow_patterns=["*.safetensors*", "*.json", "*.txt", "*.model", "*.py", "*.jinja"],
    )
    print(f"\ndownloaded in {time.time() - t0:.0f}s -> {local_dir}")

    total = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, fs in os.walk(local_dir) for f in fs
    )
    print(f"on disk: {total / 1e9:.1f} GB")
    assert any(f.endswith(".jinja") for f in os.listdir(local_dir)), "chat template missing"
else:
    print("skipped — GGUF variant selected")

## 5. Load the model

Gemma 4 is multimodal (vision **and** audio), so the entry points are `AutoProcessor` +
`AutoModelForMultimodalLM`; the concrete class is `Gemma4UnifiedForConditionalGeneration`.

Loading reports ~8 GB allocated. That is the *packed* size and it is misleading — the decompress
hook fires on the first forward in section 6 and takes it to ~25 GB.

In [ ]:
import time, torch, transformers
from transformers import AutoProcessor

ModelCls = (getattr(transformers, "AutoModelForMultimodalLM", None)
            or getattr(transformers, "AutoModelForImageTextToText", None)
            or transformers.AutoModelForCausalLM)

if USE_TRANSFORMERS:
    print("using", ModelCls.__name__)
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = ModelCls.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")
    model.eval()

    print(f"loaded in {time.time() - t0:.0f}s -> {type(model).__name__}")
    print("dtype:", next(model.parameters()).dtype)
    if torch.cuda.is_available():
        print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB (packed; grows on first forward)")
else:
    print("skipped — GGUF variant selected")

## 6. Inference

`enable_thinking=False` keeps Gemma 4 from emitting a reasoning trace before the answer; flip it to
`True` for the trace. Tokens stream as they are produced.

The first call is slow — it pays for the one-time decompression pass.

In [ ]:
import torch
from transformers import TextStreamer


def generate(messages, max_new_tokens=512, thinking=False, stream=True, **kw):
    '''Run one turn. `messages` is a standard chat list; returns the decoded reply.'''
    try:
        inputs = processor.apply_chat_template(
            messages, tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True, enable_thinking=thinking,
        )
    except TypeError:  # this build's template doesn't take enable_thinking
        inputs = processor.apply_chat_template(
            messages, tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True,
        )
    inputs = inputs.to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    streamer = TextStreamer(processor.tokenizer, skip_prompt=True,
                            skip_special_tokens=True) if stream else None
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             streamer=streamer, **kw)
    return processor.decode(out[0][input_len:], skip_special_tokens=True)


messages = [
    {"role": "system", "content": "You are a concise research assistant."},
    {"role": "user", "content": "In three sentences, explain what quantization-aware "
                               "training buys you over post-training quantization."},
]
if USE_TRANSFORMERS:
    reply = generate(messages, max_new_tokens=256)
    print(f"\n\npeak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 7. Multi-turn, sampling, and images

In [ ]:
history = [{"role": "system", "content": "You are a concise research assistant."}]


def chat(text, **kw):
    history.append({"role": "user", "content": text})
    reply = generate(history, **kw)
    history.append({"role": "assistant", "content": reply})
    return reply


if USE_TRANSFORMERS:
    chat("Name one failure mode of int4 QAT.", max_new_tokens=128,
         do_sample=True, temperature=0.7, top_p=0.95)
    print("\n---")
    chat("How would you detect that in an eval?", max_new_tokens=192)
    print(f"\n\n({len(history)} messages in history)")

In [ ]:
# Multimodal: put image parts before text parts.
vision_messages = [
    {"role": "user", "content": [
        {"type": "image",
         "url": "https://huggingface.co/datasets/huggingface/documentation-images/"
                "resolve/main/bee.jpg"},
        {"type": "text", "text": "Describe this image in one sentence."},
    ]},
]

if USE_TRANSFORMERS:
    _ = generate(vision_messages, max_new_tokens=128)
else:
    print("skipped — GGUF variant selected (llama.cpp path below is text-only here)")

## 8. GGUF path (L4 / T4 / free tier / CPU)

~7 GB of weights through `llama-cpp-python`, and it stays ~7 GB at runtime — no decompression
hook, which is exactly why this is the route for anything under ~30 GB of VRAM. It also runs
CPU-only if you accept a few tokens/sec. Independent of sections 4–7.

The wheel build takes a couple of minutes; `CMAKE_ARGS=-DGGML_CUDA=on` is what gets you GPU offload.

In [ ]:
import os, subprocess, sys, torch

# Runs only when the transformers path was skipped. Set True to force it anywhere.
RUN_GGUF = not USE_TRANSFORMERS

if torch.cuda.is_available():
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"  # inherited by the wheel build

if RUN_GGUF:
    # subprocess rather than %pip so this works inside a conditional in any kernel
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "llama-cpp-python", "huggingface_hub"], check=True)
    print("llama-cpp-python installed")
else:
    print("skipped — transformers path already loaded the model")

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

if RUN_GGUF:
    # Don't hardcode the filename — list the repo and take the single-shard Q4_0 file.
    files = [f for f in HfApi().list_repo_files(GGUF) if f.endswith(".gguf")]
    print("available:", files)
    single = [f for f in files if "-of-" not in f] or files
    gguf_file = sorted(single, key=len)[0]

    gguf_path = hf_hub_download(GGUF, filename=gguf_file)
    print("->", gguf_path)
else:
    print("skipped")

In [ ]:
import torch

if RUN_GGUF:
    from llama_cpp import Llama

    llm = Llama(
        model_path=gguf_path,
        n_ctx=8192,
        n_gpu_layers=-1 if torch.cuda.is_available() else 0,  # -1 = offload everything
        verbose=False,
    )

    stream = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": "You are a concise research assistant."},
            {"role": "user", "content": "Explain Q4_0 quantization in two sentences."},
        ],
        max_tokens=256,
        temperature=0.7,
        stream=True,
    )
    for chunk in stream:
        print(chunk["choices"][0].get("delta", {}).get("content", ""), end="", flush=True)
    print()
else:
    print("skipped")

## 9. Free the GPU

In [ ]:
import gc, torch

for name in ("model", "llm"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")